# FASE 1: CONEXIÓN GOOGLE DRIVE Y CONFIGURACIÓN DE RUTAS

In [ ]:
from google.colab import drive
import os
import glob # Importar glob para buscar archivos por patrón

# 1. Conectamos Google Drive al entorno virtual de Colab
drive.mount('/content/drive')

# 2. Definición de ruta de acceso
drive_folder_path = '/content/drive/MyDrive/Courses/PLAN  - Coursera - Google Data Analytics Professional Certificate/8. Google Data Analytics Capstone/Cyclistic Project - Case A/data'

# Definir el patrón de búsqueda global para los archivos CSV
csv_glob_path = f"{drive_folder_path}/*-divvy-tripdata.csv"

# Inicializar la lista de archivos CSV
archivos_csv = []

# 3. Validamos la existencia de la carpeta y listamos los archivos disponibles
if os.path.exists(drive_folder_path):
    print(f"\n¡Conexión exitosa! La ruta '{drive_folder_path}' existe.")

    # Usar glob para encontrar todos los archivos que coinciden con el patrón
    archivos_csv = glob.glob(csv_glob_path)

    if len(archivos_csv) > 0:
        print(f"Archivos detectados en la ruta:")
        for file_path in archivos_csv: # Iterar sobre las rutas completas ahora
            print(f" -> {os.path.basename(file_path)}") # Imprimir solo el nombre del archivo
    else:
        print(f"No se encontraron archivos CSV que coincidan con el patrón '{csv_glob_path}' en la ruta.")
else:
    print(f"\n[ERROR]: La ruta especificada '{drive_folder_path}' no existe. Revisa la ortografía.")

# FASE 2: INSTALACIÓN DE DEPENDENCIAS E INICIALIZACIÓN DE ENTORNO

In [ ]:
# Instalamos DuckDB
!pip install duckdb --quiet
# el comando quiet elimina el registro de instalación

import duckdb

# Inicializamos la conexión a la base de datos en memoria RAM (Ultra rápida y gratis)
con = duckdb.connect()

print("\n¡Entorno de DuckDB inicializado y listo para el análisis!")

# FASE 3: EXPLORACIÓN INICIAL DE DATOS (EDA PRELIMINAR)

In [ ]:
import matplotlib.pyplot as plt # Importar para visualización
import seaborn as sns # Importar para visualización
import pandas as pd # Importar pandas para el DataFrame de resumen

# Si encontramos archivos, procedemos a analizarlos pasando la lista directa a DuckDB
if len(archivos_csv) > 0:
    print("\n" + "="*80)
    print("--- 1. VISTA PRELIMINAR DE LOS DATOS CRUDOS ---")
    print("="*80)
    # DuckDB puede leer una lista de rutas de Python directamente como si fuera un array
    df_muestra = con.execute(f"SELECT * FROM read_csv_auto({archivos_csv}) LIMIT 5").df()
    display(df_muestra)

    print("\n" + "="*80)
    print("--- 2. ESQUEMA Y TIPOS DE DATOS DE LAS COLUMNAS ---")
    print("="*80)
    df_describe = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto({archivos_csv})").df()
    display(df_describe)

    print("\n" + "="*80)
    print("--- 3. TABLA RESUMEN DE VALORES ÚNICOS POR COLUMNA ---")
    print("="*80)

    # Lista para almacenar los resultados de los conteos únicos
    unique_counts_data = []

    # Iterar sobre las columnas obtenidas del esquema
    for index, row in df_describe.iterrows():
        col_name = row['column_name']
        # Escapar el nombre de la columna si contiene caracteres especiales o es una palabra clave SQL
        escaped_col_name = f'"{col_name}"'

        # Construir la consulta para contar valores únicos (excluyendo NULOS)
        query_unique_count = f"SELECT COUNT(DISTINCT {escaped_col_name}) FROM read_csv_auto({archivos_csv}) WHERE {escaped_col_name} IS NOT NULL"

        try:
            unique_count = con.execute(query_unique_count).fetchone()[0]
            unique_counts_data.append({'Columna': col_name, 'Valores Únicos': unique_count})
        except Exception as e:
            # Manejar columnas que puedan causar errores (ej. tipos no compatibles con DISTINCT)
            unique_counts_data.append({'Columna': col_name, 'Valores Únicos': f'Error: {e}'})

    # Crear un DataFrame de Pandas con los resultados
    df_unique_counts = pd.DataFrame(unique_counts_data)
    display(df_unique_counts)


    print("\n" + "="*80)
    print("--- 4. AUDITORÍA DE CALIDAD: CONTEO DE FILAS Y NULOS ORIGINALES ---")
    print("="*80)
    df_calidad = con.execute(f"""
        SELECT
            COUNT(*) AS total_registros_sucios,
            COUNT(DISTINCT ride_id) AS IDs_unicos,
            SUM(CASE WHEN start_station_name IS NULL THEN 1 ELSE 0 END) AS nulos_estacion_inicio,
            SUM(CASE WHEN end_station_name IS NULL THEN 1 ELSE 0 END) AS nulos_estacion_fin
        FROM read_csv_auto({archivos_csv})
    """).df()
    display(df_calidad)
else:
    print("\n[ERROR]: La lista 'archivos_csv' está vacía o no se ha inicializado.")
    print("Por favor, revisa la FASE 1 para asegurarte de que Python detecte tus archivos en Drive.")

# FASE 4: PIPELINE DE PROCESAMIENTO Y LIMPIEZA (ETL)

In [ ]:
if 'archivos_csv' in locals() and len(archivos_csv) > 0:
    print("\n" + "="*80)
    print("--- 1. EJECUTANDO PIPELINE ETL (UNIFICACIÓN Y LIMPIEZA) ---")
    print("="*80)
    print(f"Procesando {len(archivos_csv)} archivos en memoria... Por favor espera un momento.")

    # Usamos un query estándar de SQL y pasamos la variable usando el signo '?' para evitar BinderErrors
    con.execute("""
        CREATE OR REPLACE TABLE cyclistic_trips_cleaned AS
        SELECT
            ride_id,
            rideable_type,
            started_at,
            ended_at,

            -- Cálculo de la duración del viaje transformada limpiamente a minutos
            epoch(ended_at - started_at) / 60 AS ride_length,

            -- Extracción del día de la semana (0 = Domingo, 6 = Sábado)
            dayofweek(started_at) AS day_of_week,

            start_station_name,
            end_station_name,
            member_casual
        FROM read_csv_auto(?)

        -- Filtros de control de calidad para resguardar la integridad estadística
        WHERE ended_at > started_at                            -- Elimina inconsistencias de tiempo
          AND (epoch(ended_at - started_at) / 60) >= 1         -- Quita viajes menores a 1 min (falsos inicios)
          AND (epoch(ended_at - started_at) / 60) <= 1440      -- Quita viajes mayores a 24 hrs (robos o descuidos)
          AND start_station_name IS NOT NULL                   -- Asegura consistencia de origen
          AND end_station_name IS NOT NULL;                    -- Asegura consistencia de destino
    """, [archivos_csv])

    # Extraemos y formateamos el total de registros limpios obtenidos
    total_rows = con.execute("SELECT COUNT(*) FROM cyclistic_trips_cleaned").fetchone()[0]
    print(f"\n¡Pipeline completado con éxito!")
    print(f"Total de registros almacenados en 'cyclistic_trips_cleaned': {total_rows:,} filas.")
else:
    print("\n[ERROR]: La lista 'archivos_csv' está vacía o no se ha inicializado.")
    print("Por favor, vuelve a ejecutar la Celda 2 para asegurarte de que Python detecte tus archivos en Drive.")

## 4.1: VISUALIZACIÓN DE LA TABLA LIMPIA FINAL

In [ ]:
print("\n" + "="*80)
print("--- VISTA PREVIA DE LA TABLA 'cyclistic_trips_cleaned' ---")
print("="*80)
df_cleaned = con.execute("SELECT * FROM cyclistic_trips_cleaned LIMIT 5").df()
display(df_cleaned)

print("\n" + "="*80)
print("--- ESQUEMA DE LA TABLA 'cyclistic_trips_cleaned' ---")
print("="*80)
df_cleaned_schema = con.execute("DESCRIBE cyclistic_trips_cleaned").df()
display(df_cleaned_schema)

# Conteo de nulos en estaciones (Gobernanza Geográfica)
nulos_estaciones = con.execute("""
    SELECT COUNT(*)
    FROM read_csv_auto(?)
    WHERE start_station_name IS NULL OR end_station_name IS NULL
""", [archivos_csv]).fetchone()[0]

# Conteo de viajes menores a 1 minuto (Falsos inicios)
viajes_cortos = con.execute("""
    SELECT COUNT(*)
    FROM read_csv_auto(?)
    WHERE (epoch(ended_at - started_at) / 60) < 1
       OR ended_at <= started_at
""", [archivos_csv]).fetchone()[0]

# Conteo de viajes mayores a 24 horas (Outliers / Descuidos)
viajes_largos = con.execute("""
    SELECT COUNT(*)
    FROM read_csv_auto(?)
    WHERE (epoch(ended_at - started_at) / 60) > 1440
""", [archivos_csv]).fetchone()[0]

print(f"❌ Registros con estaciones nulas: {nulos_estaciones:,}")
print(f"⏱️ Viajes menores a 1 min (o inconsistentes): {viajes_cortos:,}")
print(f"🚲 Viajes mayores a 24 horas: {viajes_largos:,}")

# FASE 5: ANÁLISIS ESTADÍSTICO Y CONSULTAS DE NEGOCIO

In [ ]:
print("="*80)
print("--- 1. MÉTRICAS GLOBALES DE DURACIÓN POR TIPO DE USUARIO ---")
print("="*80)
# Calculamos promedio, mediana, viaje máximo y volumen total por segmento
df_duracion = con.execute("""
    SELECT
        member_casual,
        ROUND(AVG(ride_length), 2) AS promedio_duracion_minutos,
        ROUND(MEDIAN(ride_length), 2) AS mediana_duracion_minutos,
        ROUND(MAX(ride_length), 2) AS maximo_duracion_minutos,
        COUNT(ride_id) AS total_viajes,
        ROUND(COUNT(ride_id) * 100.0 / (SELECT COUNT(*) FROM cyclistic_trips_cleaned), 2) AS porcentaje_del_total
    FROM cyclistic_trips_cleaned
    GROUP BY member_casual;
""").df()
display(df_duracion)


print("\n" + "="*80)
print("--- 2. USO DEL SERVICIO POR DÍA DE LA SEMANA ---")
print("="*80)
# Mapeamos la actividad semanal (Nota: 0 = Domingo, 6 = Sábado en DuckDB)
df_semanal = con.execute("""
    SELECT
        member_casual,
        day_of_week,
        COUNT(ride_id) AS cantidad_viajes,
        ROUND(AVG(ride_length), 2) AS promedio_duracion_minutos
    FROM cyclistic_trips_cleaned
    GROUP BY member_casual, day_of_week
    ORDER BY member_casual, day_of_week;
""").df()

# Para facilitar la lectura en tu bitácora, mapeamos los números a nombres de días
dias_map = {0: '0. Domingo', 1: '1. Lunes', 2: '2. Martes', 3: '3. Miércoles', 4: '4. Jueves', 5: '5. Viernes', 6: '6. Sábado'}
df_semanal['dia_nombre'] = df_semanal['day_of_week'].map(dias_map)
df_semanal = df_semanal.sort_values(['member_casual', 'day_of_week'])
display(df_semanal[['member_casual', 'dia_nombre', 'cantidad_viajes', 'promedio_duracion_minutos']])


print("\n" + "="*80)
print("--- 3. PREFERENCIA DE TIPO DE BICICLETA ---")
print("="*80)
# Evaluamos si la elección de flota varía entre los segmentos
df_flota = con.execute("""
    SELECT
        member_casual,
        rideable_type,
        COUNT(ride_id) AS total_viajes
    FROM cyclistic_trips_cleaned
    GROUP BY member_casual, rideable_type
    ORDER BY member_casual, total_viajes DESC;
""").df()
display(df_flota)

print("\n" + "="*80)
print("--- 4. TOTAL DE VIAJES POR MES Y TIPO DE MIEMBRO ---")
print("="*80)
#Calculamos en la tabla el total de viajes por mes agrupado por tipo de miembro

df_temporada = con.execute("""
    SELECT
        member_casual,
        EXTRACT(MONTH FROM started_at) AS mes,
        COUNT(ride_id) AS cantidad_viajes
    FROM cyclistic_trips_cleaned
    GROUP BY member_casual, mes
    ORDER BY member_casual, mes;
""").df()
display(df_temporada)

## 5.1: ANÁLISIS SEMANAL Y GRÁFICOS

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configuración estilos de gráficos
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 6))

# ------------------------------------------------------------------------------
# GRÁFICO 1: Volumen de Viajes por Día de la Semana
# ------------------------------------------------------------------------------
plt.subplot(1, 2, 1)
sns.barplot(
    data=df_semanal,
    x='dia_nombre',
    y='cantidad_viajes',
    hue='member_casual',
    palette=['#FF9999', '#66B3FF'] # Colores distintivos para casual vs member
)
plt.title('Total de Viajes por Día de la Semana', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Día de la Semana', fontsize=12)
plt.ylabel('Cantidad de Viajes (Millones)', fontsize=12)
plt.xticks(rotation=45)

# ------------------------------------------------------------------------------
# GRÁFICO 2: Duración Promedio del Viaje por Día de la Semana
# ------------------------------------------------------------------------------
plt.subplot(1, 2, 2)
sns.barplot(
    data=df_semanal,
    x='dia_nombre',
    y='promedio_duracion_minutos',
    hue='member_casual',
    palette=['#FF9999', '#66B3FF']
)
plt.title('Duración Promedio del Viaje por Día', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Día de la Semana', fontsize=12)
plt.ylabel('Duración Promedio (Minutos)', fontsize=12)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 5.2: ANÁLISIS DE ESTACIONALDIAD Y GRÁFICOS

In [ ]:

print("="*80)
print("--- 1. ANÁLISIS DE ESTACIONALIDAD MENSUAL ---")
print("="*80)
# Extraemos el mes de la fecha para ver cómo evoluciona el comportamiento en el año
df_mensual = con.execute("""
    SELECT
        member_casual,
        EXTRACT(MONTH FROM started_at) AS mes,
        COUNT(ride_id) AS cantidad_viajes
    FROM cyclistic_trips_cleaned
    GROUP BY member_casual, mes
    ORDER BY mes, member_casual;
""").df()

# Mapeamos los números a nombres de meses para el reporte
meses_map = {1: 'Ene', 2: 'Feb', 3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun',
             7: 'Jul', 8: 'Ago', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dic'}
df_mensual['mes_nombre'] = df_mensual['mes'].map(meses_map)

# Graficamos la estacionalidad
plt.figure(figsize=(12, 5))
sns.lineplot(data=df_mensual, x='mes_nombre', y='cantidad_viajes', hue='member_casual', marker='o', linewidth=2.5)
plt.title('Evolución Mensual de Viajes (Estacionalidad)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Mes del Año', fontsize=12)
plt.ylabel('Total de Viajes', fontsize=12)
plt.show()


print("\n" + "="*80)
print("--- 2. TOP 10 ESTACIONES DE INICIO MÁS POPULARES PARA CASUALES ---")
print("="*80)
# Identificamos los "Hotspots" geográficos estratégicos para el negocio
df_top_estaciones_casual = con.execute("""
    SELECT
        start_station_name,
        COUNT(ride_id) AS cantidad_viajes
    FROM cyclistic_trips_cleaned
    WHERE member_casual = 'casual'
    GROUP BY start_station_name
    ORDER BY cantidad_viajes DESC
    LIMIT 10;
""").df()
display(df_top_estaciones_casual)

## 5.3: EXPORTACIÓN DE TABLAS RESUMEN PARA DASHBOARD (BI)

In [ ]:
print("\n" + "="*80)
print("--- 3. EXPORTANDO RESUMEN COMPACTO A TU GOOGLE DRIVE ---")
print("="*80)

# Agrupamos toda nuestra inteligencia de negocio en una sola tabla agregada y compacta
df_dashboard_ready = con.execute("""
    SELECT
        member_casual,
        rideable_type,
        day_of_week,
        EXTRACT(MONTH FROM started_at) AS mes,
        start_station_name,
        COUNT(ride_id) AS total_viajes,
        ROUND(AVG(ride_length), 2) AS promedio_duracion_minutos
    FROM cyclistic_trips_cleaned
    GROUP BY member_casual, rideable_type, day_of_week, mes, start_station_name, end_station_name;
""").df()

# Guardamos el archivo directamente en la misma carpeta de tus datos en Drive
output_path = os.path.join(drive_folder_path, 'cyclistic_summary_dashboard.csv')
df_dashboard_ready.to_csv(output_path, index=False)

print(f"¡Éxito total! Archivo de resumen exportado listo para Tableau/Power BI.")
print(f"Ruta en tu Drive: {output_path}")
display(df_dashboard_ready)

## 5.3: VISUALIZACIÓN DE HOTSPOTS GEOGRÁFICOS (TARGET: CASUALS)


In [ ]:
plt.figure(figsize=(12, 6))

# Creamos un gráfico de barras horizontales para el Top 10 de estaciones
sns.barplot(
    data=df_top_estaciones_casual,
    x='cantidad_viajes',
    y='start_station_name',
    palette='Reds_r' # Degradado rojo para simular un mapa de calor urbano
)

plt.title('Top 10 Estaciones de Inicio con Mayor Tráfico Casual', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Cantidad Total de Viajes (Últimos 12 Meses)', fontsize=12)
plt.ylabel('Nombre de la Estación', fontsize=12)

# Añadimos las etiquetas de datos al final de cada barra para máxima claridad
for index, value in enumerate(df_top_estaciones_casual['cantidad_viajes']):
    plt.text(value, index, f' {value:,}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()